# Chapter 6, Exercise 2: Full fine-tuning versus LoRA parameter counts

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 6, Exercise 2.** For an encoder with about 300 million parameters, estimate the trainable-parameter count for full fine-tuning versus a LoRA update of rank 8 applied to the attention query and value projections, and give the approximate ratio. Use the approximation of 2dr trainable parameters per adapted projection, where d is the hidden size and r is the LoRA rank, and state your assumptions.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


## 1. Assumptions

A 300 M-parameter speech encoder in the wav2vec 2.0 / HuBERT "Large" family has **24 Transformer layers** with hidden size **d = 1024** (16 heads of 64). Each layer has one query and one value projection, both square d × d matrices. LoRA replaces the update of each adapted matrix by the product of two thin matrices of shapes d × r and r × d, so each adapted projection adds **2 d r** trainable parameters (bias terms and the tiny task head are ignored).

In [1]:
P_total = 300e6      # encoder parameters
d, r, L = 1024, 8, 24  # hidden size, LoRA rank, number of layers
projections_per_layer = 2   # query and value

per_projection = 2 * d * r
lora_params = per_projection * projections_per_layer * L
print(f"per adapted projection : 2 x {d} x {r} = {per_projection:,}")
print(f"per layer (Q and V)    : {per_projection*projections_per_layer:,}")
print(f"LoRA total over {L} layers: {lora_params:,}  (about {lora_params/1e6:.2f} M)")
print(f"full fine-tuning       : {P_total:,.0f}")
print(f"ratio full : LoRA      = {P_total/lora_params:,.0f} : 1")
print(f"LoRA as a fraction of the encoder: {100*lora_params/P_total:.2f} %")

per adapted projection : 2 x 1024 x 8 = 16,384
per layer (Q and V)    : 32,768
LoRA total over 24 layers: 786,432  (about 0.79 M)
full fine-tuning       : 300,000,000
ratio full : LoRA      = 381 : 1
LoRA as a fraction of the encoder: 0.26 %


## 2. Result

| Setting | Trainable parameters |
|---|---|
| Full fine-tuning | about 300,000,000 (every weight) |
| LoRA, r = 8, Q and V, 24 layers of d = 1024 | 2 × 1024 × 8 = 16,384 per projection; × 2 projections × 24 layers = **786,432 (about 0.79 M)** |
| Ratio | roughly **380 : 1**, i.e. LoRA trains about **0.26 %** of the parameters |

## 3. Sensitivity to the assumptions

The count scales linearly with rank, with the number of adapted projections and with depth. The cell below shows a few alternatives so that the reader can match a different encoder (for example a 12-layer, d = 768 "Base" model, or adapting all four attention projections).

In [2]:
import pandas as pd
rows = []
for (d_, L_, name) in [(1024, 24, "Large-type (300 M)"), (768, 12, "Base-type (95 M)")]:
    for proj in [2, 4]:
        for r_ in [4, 8, 16, 64]:
            n = 2*d_*r_*proj*L_
            rows.append({"encoder": name, "adapted projections": proj, "rank r": r_,
                         "LoRA params": n, "% of 300 M": round(100*n/300e6, 3)})
pd.DataFrame(rows)

,encoder,adapted projections,rank r,LoRA params,% of 300 M
0,Large-type (300 M),2,4,393216,0.131
1,Large-type (300 M),2,8,786432,0.262
2,Large-type (300 M),2,16,1572864,0.524
3,Large-type (300 M),2,64,6291456,2.097
4,Large-type (300 M),4,4,786432,0.262
5,Large-type (300 M),4,8,1572864,0.524
6,Large-type (300 M),4,16,3145728,1.049
7,Large-type (300 M),4,64,12582912,4.194
8,Base-type (95 M),2,4,147456,0.049
9,Base-type (95 M),2,8,294912,0.098


**Caveats the report should state.** (1) The fraction refers to *trainable* parameters; the full frozen backbone must still be loaded in memory, so LoRA saves optimizer state and gradient memory, not activation memory. (2) Fewer trainable parameters does not by itself mean less labeled data or shorter training, and LoRA does not match full fine-tuning in every setting (Section 6.8, Table 6.2). (3) A per-dialect LoRA of under 1 M parameters (a few megabytes) is what makes storing one adaptation per Arabic dialect on a shared backbone practical.